In [2]:
import random
import math

def asignacion_aleatoria(trabajos, m):
  cargas = [0] * m
  asignaciones = []
  for w in trabajos:
    srv = random.randrange(m)
    cargas[srv] += w
    asignaciones.append(srv)
  return cargas, asignaciones


def poder_de_dos_opciones(trabajos, m):
  cargas = [0] * m
  asignaciones = []
  for w in trabajos:
    a, b = random.randrange(m), random.randrange(m)
    srv = a if cargas[a] <= cargas[b] else b
    cargas[srv] += w
    asignaciones.append(srv)
  return cargas, asignaciones


def poder_de_d_opciones(trabajos, m, d=3):
    cargas = [0] * m
    asignaciones = []
    for w in trabajos:
        candidatos = random.sample(range(m), min(d,m))
        srv = min(candidatos, key = lambda s: cargas[s])
        cargas[srv] += w
        asignaciones.append(srv)
    return cargas, asignaciones


def round_robin(trabajos, m):
  cargas = [0] * m
  asignaciones = []
  for i, w in enumerate(trabajos):
      srv = i % m
      cargas[srv] += w
      asignaciones.append(srv)
  return cargas, asignaciones


def makespan(cargas):
  return max(cargas)


def desbalance(cargas):
  avg = sum(cargas) / len(cargas)
  return (max(cargas) - avg) / avg * 100 if avg > 0 else 0


def comparar(m=8, n=200, wmax=10, trials=1000):
    resultados = {
        'aleatorio': [],
        'two_choices': [],
        'three_choices': [],
        'round_robin': []
    }

    for _ in range(trials):
        jobs = [random.randint(1, wmax) for _ in range(n)]
        resultados['aleatorio'].append(
            makespan(asignacion_aleatoria(jobs, m)[0]))
        resultados['two_choices'].append(
            makespan(poder_de_dos_opciones(jobs, m)[0]))
        resultados['three_choices'].append(
            makespan(poder_de_d_opciones(jobs, m, d=3)[0]))
        resultados['round_robin'].append(
            makespan(round_robin(jobs, m)[0]))

    total = sum([random.randint(1, wmax) for _ in range(n)])
    opt = math.ceil(total / m)  # cota inferior del óptimo

    print(f"m={m} servidores | n={n} trabajos | w∈[1,{wmax}] | {trials} trials\n")
    print(f"{'Algoritmo':<20} {'Makespan prom':>15} {'Makespan máx':>13} {'vs OPT':>8}")
    print("─" * 60)
    for nombre, vals in resultados.items():
        avg = sum(vals) / len(vals)
        mx  = max(vals)
        print(f"{nombre:<20} {avg:>15.1f} {mx:>13} {avg/opt:>7.2f}x")

    print(f"\n  OPT (cota inferior) ≈ {opt}")


# ───────────────────────
# Demo paso a paso
# ───────────────────────

if __name__ == "__main__":
    random.seed(42)
    m, n = 4, 12
    jobs = [random.randint(1, 8) for _ in range(n)]

    print(f"Trabajos: {jobs}\n")

    for nombre, fn in [
        ("Aleatorio simple", asignacion_aleatoria),
        ("Two choices",      poder_de_dos_opciones),
    ]:
        cargas, asig = fn(jobs, m)
        print(f"{nombre}:")
        for i, c in enumerate(cargas):
            barra = "█" * c
            print(f"  S{i+1}: {barra:<30} {c}")
        print(f"  Makespan: {makespan(cargas)}  |  Desbalance: {desbalance(cargas):.1f}%\n")

    print("─" * 60)
    comparar(m=6, n=500, wmax=10, trials=2000)


Trabajos: [2, 1, 5, 4, 4, 3, 2, 2, 7, 1, 1, 2]

Aleatorio simple:
  S1: ████████████                   12
  S2: ███████████                    11
  S3: ████                           4
  S4: ███████                        7
  Makespan: 12  |  Desbalance: 41.2%

Two choices:
  S1: ██████████                     10
  S2: ███                            3
  S3: █████████████                  13
  S4: ████████                       8
  Makespan: 13  |  Desbalance: 52.9%

────────────────────────────────────────────────────────────
m=6 servidores | n=500 trabajos | w∈[1,10] | 2000 trials

Algoritmo              Makespan prom  Makespan máx   vs OPT
────────────────────────────────────────────────────────────
aleatorio                      531.9           658    1.17x
two_choices                    465.7           504    1.03x
three_choices                  462.8           502    1.02x
round_robin                    491.9           574    1.09x

  OPT (cota inferior) ≈ 453
